# Build talking-head timeline from small PNG frames

This notebook creates a reusable frame timeline from word timings.

Rules implemented:
- During speaking intervals: alternate mouth open/closed in a natural rhythm.
- During silence gaps: force mouth closed.
- Optional natural blinks during silence.

Primary output:
- `timeline_png_25fps/anim_0001.png`, `anim_0002.png`, ... (25 fps)

Optional side outputs:
- `timeline_small_frames.csv` (one row per video frame)
- `timeline_small_frames.ffconcat` (ready for ffmpeg concat)


In [1]:
from __future__ import annotations

import csv
import math
import random
import re
import shutil
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, List


@dataclass
class Settings:
    frames_dir: Path = Path("frames_small")
    timings_file: Path = Path("timings.txt")
    fps: int = 25
    rng_seed: int = 7

    # Keep blinks rare so eyes are open most of the time.
    blink_speaking_every_s_min: float = 6.0
    blink_speaking_every_s_max: float = 11.0
    blink_silence_every_s_min: float = 5.0
    blink_silence_every_s_max: float = 9.0
    blink_len_frames: int = 2

    png_timeline_dir: Path = Path("timeline_png_25fps")
    png_prefix: str = "anim"
    timeline_csv: Path = Path("timeline_small_frames.csv")
    ffconcat_file: Path = Path("timeline_small_frames.ffconcat")


SETTINGS = Settings()
SETTINGS


Settings(frames_dir=WindowsPath('frames_small'), timings_file=WindowsPath('timings.txt'), fps=25, rng_seed=7, blink_speaking_every_s_min=6.0, blink_speaking_every_s_max=11.0, blink_silence_every_s_min=5.0, blink_silence_every_s_max=9.0, blink_len_frames=2, png_timeline_dir=WindowsPath('timeline_png_25fps'), png_prefix='anim', timeline_csv=WindowsPath('timeline_small_frames.csv'), ffconcat_file=WindowsPath('timeline_small_frames.ffconcat'))

In [2]:
@dataclass
class SpeechSpan:
    token: str
    start_s: float
    end_s: float


def parse_timings(path: Path) -> List[SpeechSpan]:
    # Supports lines like: "word: 1.24s - 1.40s", "word: 1.24 s – 1.40 s"
    pattern = re.compile(r"^\s*(.*?)\s*:\s*([0-9]*\.?[0-9]+)\s*s\s*[\-–]\s*([0-9]*\.?[0-9]+)\s*s\s*$")
    spans: List[SpeechSpan] = []
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line:
            continue
        m = pattern.match(line)
        if not m:
            raise ValueError(f"Could not parse timing line: {line}")
        token, start_s, end_s = m.group(1), float(m.group(2)), float(m.group(3))
        if end_s < start_s:
            raise ValueError(f"End before start in line: {line}")
        spans.append(SpeechSpan(token=token, start_s=start_s, end_s=end_s))
    if not spans:
        raise ValueError("No timings found.")
    return spans


def find_frame_path(frames_dir: Path, required_keywords: Iterable[str]) -> Path:
    keywords = [k.lower() for k in required_keywords]
    candidates = sorted(frames_dir.glob("*.png"))
    for p in candidates:
        stem = p.stem.lower()
        if all(k in stem for k in keywords):
            return p
    raise FileNotFoundError(
        f"No frame in {frames_dir} matched keywords {keywords}. "
        f"Available: {[c.name for c in candidates]}"
    )


def is_speaking(t: float, spans: List[SpeechSpan]) -> bool:
    return any(s.start_s <= t < s.end_s for s in spans)


def make_timeline(settings: Settings) -> list[dict]:
    spans = parse_timings(settings.timings_file)
    rng = random.Random(settings.rng_seed)

    open_frame = find_frame_path(settings.frames_dir, ["mouth", "open"])
    closed_frame = find_frame_path(settings.frames_dir, ["mouth", "closed"])
    blink_frame = find_frame_path(settings.frames_dir, ["blink"])

    total_duration_s = max(s.end_s for s in spans)
    frame_duration_s = 1.0 / settings.fps
    total_frames = int(math.ceil(total_duration_s * settings.fps))

    # Two independent blink schedulers: blinks can happen in both speaking and silence.
    next_blink_speaking = rng.randint(
        int(settings.blink_speaking_every_s_min * settings.fps),
        int(settings.blink_speaking_every_s_max * settings.fps),
    )
    next_blink_silence = rng.randint(
        int(settings.blink_silence_every_s_min * settings.fps),
        int(settings.blink_silence_every_s_max * settings.fps),
    )
    blink_remaining = 0

    mouth_is_open = False
    mouth_hold_remaining = 0

    rows: list[dict] = []

    for i in range(total_frames):
        t = i * frame_duration_s
        speaking = is_speaking(t, spans)

        if blink_remaining > 0:
            frame_path = blink_frame
            blink_remaining -= 1
        else:
            should_blink = (speaking and i >= next_blink_speaking) or ((not speaking) and i >= next_blink_silence)

            if should_blink:
                frame_path = blink_frame
                blink_remaining = max(0, settings.blink_len_frames - 1)

                if speaking:
                    next_blink_speaking = i + rng.randint(
                        int(settings.blink_speaking_every_s_min * settings.fps),
                        int(settings.blink_speaking_every_s_max * settings.fps),
                    )
                else:
                    next_blink_silence = i + rng.randint(
                        int(settings.blink_silence_every_s_min * settings.fps),
                        int(settings.blink_silence_every_s_max * settings.fps),
                    )
            elif speaking:
                # Alternate open/closed with short holds for natural speech.
                if mouth_hold_remaining <= 0:
                    mouth_is_open = not mouth_is_open
                    mouth_hold_remaining = rng.randint(1, 2) if mouth_is_open else rng.randint(1, 3)
                frame_path = open_frame if mouth_is_open else closed_frame
                mouth_hold_remaining -= 1
            else:
                # During silence: keep mouth closed unless this frame is a blink.
                mouth_is_open = False
                mouth_hold_remaining = 0
                frame_path = closed_frame

        rows.append(
            {
                "frame_idx": i,
                "time_s": round(t, 6),
                "speaking": int(speaking),
                "image": frame_path.as_posix(),
                "duration_s": round(frame_duration_s, 6),
            }
        )

    blink_str = blink_frame.as_posix()

    def ensure_blink_in_state(state_speaking: int) -> None:
        state_indices = [idx for idx, row in enumerate(rows) if row["speaking"] == state_speaking]
        if not state_indices:
            return

        has_blink = any(rows[idx]["image"] == blink_str for idx in state_indices)
        if has_blink:
            return

        # Guarantee at least one blink in both speaking and silence timelines.
        start = state_indices[len(state_indices) // 2]
        for offset in range(settings.blink_len_frames):
            idx = start + offset
            if idx >= len(rows) or rows[idx]["speaking"] != state_speaking:
                break
            rows[idx]["image"] = blink_str

    ensure_blink_in_state(1)
    ensure_blink_in_state(0)

    return rows


def save_csv(rows: list[dict], csv_path: Path) -> None:
    if not rows:
        raise ValueError("No timeline rows to save")
    with csv_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def save_ffconcat(rows: list[dict], ffconcat_path: Path) -> None:
    if not rows:
        raise ValueError("No timeline rows to save")

    lines = ["ffconcat version 1.0"]
    for row in rows:
        lines.append(f"file {row['image']}")
        lines.append(f"duration {row['duration_s']}")
    # ffmpeg concat demuxer expects final file repeated once.
    lines.append(f"file {rows[-1]['image']}")

    ffconcat_path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def save_png_timeline(rows: list[dict], out_dir: Path, prefix: str = "anim") -> None:
    if not rows:
        raise ValueError("No timeline rows to save")

    out_dir.mkdir(parents=True, exist_ok=True)

    # Keep only previously generated timeline frames (same prefix + 4 digits).
    for old in out_dir.glob(f"{prefix}_[0-9][0-9][0-9][0-9].png"):
        old.unlink()

    for i, row in enumerate(rows, start=1):
        src = Path(row["image"])
        dst = out_dir / f"{prefix}_{i:04d}.png"
        shutil.copy2(src, dst)


In [3]:
rows = make_timeline(SETTINGS)
save_png_timeline(rows, SETTINGS.png_timeline_dir, SETTINGS.png_prefix)

# Optional side outputs (kept for compatibility)
save_csv(rows, SETTINGS.timeline_csv)
save_ffconcat(rows, SETTINGS.ffconcat_file)

total = len(rows)
speaking_count = sum(r["speaking"] for r in rows)
silent_count = total - speaking_count
open_count = sum(1 for r in rows if r["image"].endswith("mouth_open.png"))
blink_count = sum(1 for r in rows if r["image"].endswith("blink.png"))
blink_speaking = sum(1 for r in rows if r["speaking"] == 1 and r["image"].endswith("blink.png"))
blink_silent = sum(1 for r in rows if r["speaking"] == 0 and r["image"].endswith("blink.png"))
closed_or_blink = total - open_count

print(f"FPS: {SETTINGS.fps}")
print(f"Frames: {total}")
print(f"Speaking frames: {speaking_count}")
print(f"Silent frames: {silent_count} (mouth closed except blink)")
print(f"Open-mouth frames: {open_count}")
print(f"Blink frames total: {blink_count}")
print(f"Blink frames while speaking: {blink_speaking}")
print(f"Blink frames while silent: {blink_silent}")
print(f"Closed/blink frames: {closed_or_blink}")
print(f"PNG timeline dir: {SETTINGS.png_timeline_dir.as_posix()}")
print(f"Example filename: {SETTINGS.png_prefix}_0001.png")
print(f"CSV (optional): {SETTINGS.timeline_csv.as_posix()}")
print(f"FFConcat (optional): {SETTINGS.ffconcat_file.as_posix()}")

rows[:12]


FPS: 25
Frames: 279
Speaking frames: 221
Silent frames: 58 (mouth closed except blink)
Open-mouth frames: 98
Blink frames total: 4
Blink frames while speaking: 2
Blink frames while silent: 2
Closed/blink frames: 181
PNG timeline dir: timeline_png_25fps
Example filename: anim_0001.png
CSV (optional): timeline_small_frames.csv
FFConcat (optional): timeline_small_frames.ffconcat


[{'frame_idx': 0,
  'time_s': 0.0,
  'speaking': 1,
  'image': 'frames_small/anim_Luca_mouth_open.png',
  'duration_s': 0.04},
 {'frame_idx': 1,
  'time_s': 0.04,
  'speaking': 1,
  'image': 'frames_small/anim_Luca_mouth_open.png',
  'duration_s': 0.04},
 {'frame_idx': 2,
  'time_s': 0.08,
  'speaking': 1,
  'image': 'frames_small/anim_Luca_mouth_closed.png',
  'duration_s': 0.04},
 {'frame_idx': 3,
  'time_s': 0.12,
  'speaking': 1,
  'image': 'frames_small/anim_Luca_mouth_closed.png',
  'duration_s': 0.04},
 {'frame_idx': 4,
  'time_s': 0.16,
  'speaking': 1,
  'image': 'frames_small/anim_Luca_mouth_closed.png',
  'duration_s': 0.04},
 {'frame_idx': 5,
  'time_s': 0.2,
  'speaking': 1,
  'image': 'frames_small/anim_Luca_mouth_open.png',
  'duration_s': 0.04},
 {'frame_idx': 6,
  'time_s': 0.24,
  'speaking': 1,
  'image': 'frames_small/anim_Luca_mouth_closed.png',
  'duration_s': 0.04},
 {'frame_idx': 7,
  'time_s': 0.28,
  'speaking': 1,
  'image': 'frames_small/anim_Luca_mouth_open

## Notes

The main deliverable is a numbered PNG timeline in `timeline_png_25fps` at 25 fps.

Optional ffmpeg render path (if needed later):

```bash
ffmpeg -f concat -safe 0 -i timeline_small_frames.ffconcat -i short_story.mp3 -c:v libx264 -pix_fmt yuv420p -shortest out_small_frames.mp4
```

To reuse this notebook for another animation, update `SETTINGS` and ensure your frame filenames include these keywords:
- open-mouth frame: `mouth` + `open`
- closed-mouth frame: `mouth` + `closed`
- blink frame: `blink`
